We will clean, extend and split the google/granola-entity-questions dataset

In [1]:
from collections import defaultdict

from datasets import load_dataset, DatasetDict, Dataset

In [2]:
data = load_dataset("google/granola-entity-questions", split="train")

In [3]:
answers_fields = [f"granola_answer_{i}" for i in range(1, 15)]

def normalize_entry(entry):
    for field in answers_fields:
        val = entry.get(field, None)
        entry[field] = "" if val is None else str(val)
    return entry

data = [normalize_entry(e) for e in data]

In [4]:
def count_granolas():
    count_dict = defaultdict(int)
    for entry in data:
        for i, field in enumerate(answers_fields):
            if entry[field]:
                count_dict[field] += 1
    return count_dict

def count_granolas_exact(data, answers_fields):
    count_dict = defaultdict(int)

    for entry in data:
        for field in reversed(answers_fields):
            if entry.get(field):
                level = int(field.split("_")[-1])
                count_dict[level] += 1
                break

    return dict(count_dict)

print(count_granolas())
print(count_granolas_exact(data, answers_fields)) #12452

defaultdict(<class 'int'>, {'granola_answer_1': 12452, 'granola_answer_2': 12131, 'granola_answer_3': 9602, 'granola_answer_4': 1890, 'granola_answer_5': 143, 'granola_answer_6': 27, 'granola_answer_7': 10, 'granola_answer_8': 9, 'granola_answer_9': 4, 'granola_answer_10': 4})
{4: 1749, 3: 7719, 2: 2611, 5: 116, 1: 230, 6: 17, 10: 4, 8: 5, 7: 1}


In [5]:
duplicate_answers = 0
duplicate_entries = []
for entry in data:
    answer_set = set()
    answer_in_entry = 0
    for i, field in enumerate(answers_fields):
        if entry[field]:
            answer_in_entry += 1
            answer_set.add(entry[field])
        if len(answer_set) != answer_in_entry:
            duplicate_answers += 1
            duplicate_entries.append(entry)
duplicate_answers

267

In [6]:
duplicate_entries[0]

{'Unnamed: 0': 2471,
 'relation': 'P26',
 'question': 'Who is James Joyce married to?',
 'question_entity': 'James Joyce',
 'question_entity_qid': 'Q6882',
 'question_entity_description': 'Irish novelist and poet (1882–1941)',
 'question_entity_popularity': 71665.0,
 'answer': 'Nora Barnacle',
 'answer_entity_qid': 'Q444609',
 'answer_entity_description': 'Wife of James Joyce',
 'answer_entity_popularity': 6693.0,
 'score_for_potential_error': 0.0,
 'granola_answer_1': 'Nora Barnacle',
 'granola_answer_2': 'Nora Barnacle Joyce',
 'granola_answer_3': 'Nora Barnacle Joyce',
 'granola_answer_4': '',
 'granola_answer_5': '',
 'granola_answer_6': '',
 'granola_answer_7': '',
 'granola_answer_8': '',
 'granola_answer_9': '',
 'granola_answer_10': '',
 'granola_answer_11': '',
 'granola_answer_12': '',
 'granola_answer_13': '',
 'granola_answer_14': '',
 'num_granola_answers': 3}

We can see, there are entries which have the exact same resposne in different granularity levels at the same time. Similar to the Granola Paper, we skip them. E.g.: ID 4376 and 2471

In [7]:
cleaned_data = []
for entry in data:
    answer_set = set()
    duplicate_answers = False
    for i, field in enumerate(answers_fields):
        if entry[field]:
            if entry[field] in answer_set:
                duplicate_answers = True
                break
            else:
                answer_set.add(entry[field])
    if not duplicate_answers:
        cleaned_data.append(entry)
data = cleaned_data

In [8]:
count_granolas()

defaultdict(int,
            {'granola_answer_1': 12429,
             'granola_answer_2': 12108,
             'granola_answer_3': 9579,
             'granola_answer_4': 1878,
             'granola_answer_5': 134,
             'granola_answer_6': 23,
             'granola_answer_7': 7,
             'granola_answer_8': 6,
             'granola_answer_9': 2,
             'granola_answer_10': 2})

In [9]:
for entry in data:
    answer_set = set()
    answer_in_entry = 0
    if entry['granola_answer_6'] and not entry['granola_answer_7']:
        print(entry)

{'Unnamed: 0': 2160, 'relation': 'P26', 'question': 'Who is Dani Calabresa married to?', 'question_entity': 'Dani Calabresa', 'question_entity_qid': 'Q5216217', 'question_entity_description': 'Brazilian actress, comidian, screenwriter, TV host and reporter', 'question_entity_popularity': 262.0, 'answer': 'Marcelo Adnet', 'answer_entity_qid': 'Q6756484', 'answer_entity_description': 'Brazilian comedian, actor, TV host, composer and carnival producer', 'answer_entity_popularity': 263.0, 'score_for_potential_error': 0.0, 'granola_answer_1': 'Marcelo Adnet', 'granola_answer_2': 'a Brazilian comedian', 'granola_answer_3': 'a Brazilian actor', 'granola_answer_4': 'a Brazilian TV host', 'granola_answer_5': 'a Brazilian composer', 'granola_answer_6': 'a Brazilian carnival producer', 'granola_answer_7': '', 'granola_answer_8': '', 'granola_answer_9': '', 'granola_answer_10': '', 'granola_answer_11': '', 'granola_answer_12': '', 'granola_answer_13': '', 'granola_answer_14': '', 'num_granola_answ

In [10]:
for entry in data:
    answer_set = set()
    answer_in_entry = 0
    if entry['granola_answer_5'] and not entry['granola_answer_6']:
        print(entry)

{'Unnamed: 0': 289, 'relation': 'P19', 'question': 'Where was Arthur Gee born?', 'question_entity': 'Arthur Gee', 'question_entity_qid': 'Q4798793', 'question_entity_description': 'English footballer (1892-1959)', 'question_entity_popularity': 18.0, 'answer': 'Earlestown', 'answer_entity_qid': 'Q1748301', 'answer_entity_description': 'town in Newton-le-Willows, St Helens, Merseyside, UK', 'answer_entity_popularity': 466.0, 'score_for_potential_error': 0.0, 'granola_answer_1': 'Earlestown', 'granola_answer_2': 'Newton-le-Willows', 'granola_answer_3': 'St Helens', 'granola_answer_4': 'Merseyside', 'granola_answer_5': 'England', 'granola_answer_6': '', 'granola_answer_7': '', 'granola_answer_8': '', 'granola_answer_9': '', 'granola_answer_10': '', 'granola_answer_11': '', 'granola_answer_12': '', 'granola_answer_13': '', 'granola_answer_14': '', 'num_granola_answers': 5}
{'Unnamed: 0': 650, 'relation': 'P19', 'question': 'Where was Jean van de Velde born?', 'question_entity': 'Jean van de

Since we only have very few samples which have > 4 granularity, and after inspection, these do not all make really sense, e.g. a composer < an actress. We get rid off them and only have a look at granularity 1 - 4.

In [11]:
cleaned_data = []
removed = 0
for entry in data:
    answer_set = set()
    answer_in_entry = 0
    if entry['granola_answer_5']:
        removed += 1
        continue
    else:
        cleaned_data.append(entry)
data = cleaned_data
removed

134

In [12]:
count_granolas()

defaultdict(int,
            {'granola_answer_1': 12295,
             'granola_answer_2': 11979,
             'granola_answer_3': 9445,
             'granola_answer_4': 1746})

In [13]:
data[3735]

{'Unnamed: 0': 4000,
 'relation': 'P50',
 'question': 'Who is the author of The Golden Fleecing?',
 'question_entity': 'The Golden Fleecing',
 'question_entity_qid': 'Q3576811',
 'question_entity_description': '1955 Uncle Scrooge comic book story by Carl Barks',
 'question_entity_popularity': 86.0,
 'answer': 'Carl Barks',
 'answer_entity_qid': 'Q11941',
 'answer_entity_description': 'American Disney comics creator (1901-2000)',
 'answer_entity_popularity': 6680.0,
 'score_for_potential_error': 0.0,
 'granola_answer_1': 'Carl Barks',
 'granola_answer_2': 'an American',
 'granola_answer_3': '',
 'granola_answer_4': '',
 'granola_answer_5': '',
 'granola_answer_6': '',
 'granola_answer_7': '',
 'granola_answer_8': '',
 'granola_answer_9': '',
 'granola_answer_10': '',
 'granola_answer_11': '',
 'granola_answer_12': '',
 'granola_answer_13': '',
 'granola_answer_14': '',
 'num_granola_answers': 2}

There are entries which have granular answer 3 but not granular answer 2. We should remove them, as this was initially not the intent in the Granola Dataset.

In [14]:
counter = 0
cleaned_data = []
for entry in data:
    answer_missing = False
    answer_skipped = False
    for i, field in enumerate(answers_fields):
        if not entry[field]:
            answer_missing = True
        if answer_missing and entry[field]:
            counter += 1
            answer_skipped = True
            continue
    if not answer_skipped:
        cleaned_data.append(entry)
data = cleaned_data
counter

115

In [15]:
count_granolas()

defaultdict(int,
            {'granola_answer_1': 12200,
             'granola_answer_2': 11970,
             'granola_answer_3': 9359,
             'granola_answer_4': 1717})

In [16]:
data[0]

{'Unnamed: 0': 0,
 'relation': 'P19',
 'question': 'Where was Luke Prokopec born?',
 'question_entity': 'Luke Prokopec',
 'question_entity_qid': 'Q6702280',
 'question_entity_description': 'Australian baseball player',
 'question_entity_popularity': 93.0,
 'answer': 'Blackwood',
 'answer_entity_qid': 'Q880986',
 'answer_entity_description': 'town and community in Caerphilly County Borough',
 'answer_entity_popularity': 2272.0,
 'score_for_potential_error': 0.0,
 'granola_answer_1': 'Blackwood',
 'granola_answer_2': 'Caerphilly County Borough',
 'granola_answer_3': 'Wales',
 'granola_answer_4': 'United Kingdom',
 'granola_answer_5': '',
 'granola_answer_6': '',
 'granola_answer_7': '',
 'granola_answer_8': '',
 'granola_answer_9': '',
 'granola_answer_10': '',
 'granola_answer_11': '',
 'granola_answer_12': '',
 'granola_answer_13': '',
 'granola_answer_14': '',
 'num_granola_answers': 4}

In the Granola Paper, they remove hard-coded responses that they defined as trivial (such as "person", "university", etc.). As we dont know the full list of them and we also can use them for granularity. We keep them.

  Now we can remove unused columns (granola > 4) and update num_granola_answers

In [17]:
data = Dataset.from_list(data)

columns_to_remove = [f"granola_answer_{i}" for i in range(5, 15)]
def update_num_answers(entry):
    counter = 0
    for i in range(1, 5):
        if entry[f"granola_answer_{i}"]:
            counter += 1
    return {'num_granola_answers': counter}

data = data.remove_columns(columns_to_remove)
data = data.map(update_num_answers)
data = data.rename_column('Unnamed: 0', 'id')

Map:   0%|          | 0/12200 [00:00<?, ? examples/s]

In [18]:
data[0]

{'id': 0,
 'relation': 'P19',
 'question': 'Where was Luke Prokopec born?',
 'question_entity': 'Luke Prokopec',
 'question_entity_qid': 'Q6702280',
 'question_entity_description': 'Australian baseball player',
 'question_entity_popularity': 93.0,
 'answer': 'Blackwood',
 'answer_entity_qid': 'Q880986',
 'answer_entity_description': 'town and community in Caerphilly County Borough',
 'answer_entity_popularity': 2272.0,
 'score_for_potential_error': 0.0,
 'granola_answer_1': 'Blackwood',
 'granola_answer_2': 'Caerphilly County Borough',
 'granola_answer_3': 'Wales',
 'granola_answer_4': 'United Kingdom',
 'num_granola_answers': 4}

We want to split our dataset without any data leakage from train to val to test. E.g. Answer 4 already in train dataset for a different entity. We take a greedy approach.

In [20]:
from tqdm import tqdm
import numpy as np
from collections import defaultdict
import random

def split_no_answer_leakage(
    dataset,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42,
    max_tries=10
):
    random.seed(seed)

    target_counts = {
        "train": int(len(dataset) * train_ratio),
        "val": int(len(dataset) * val_ratio),
        "test": int(len(dataset) * test_ratio)
    }

    def calc_target_difference(proposed_split):
        return (target_counts['train'] - len(proposed_split['train'])) + (target_counts['val'] - len(proposed_split['val'])) + (target_counts['test'] - len(proposed_split['test']))

    def get_entity_answers(entry):
        answer_fields = [f"granola_answer_{i}" for i in range(1, 5)]
        return {entry[answer_field] for answer_field in answer_fields if entry[answer_field]}

    best_try = None
    best_try_count = np.inf
    for _ in tqdm(range(max_tries)):
        dataset = dataset.shuffle()

        splits = {"train": [], "val": [], "test": []}
        answers_seen = {"train": set(), "val": set(), "test": set()}

        split_names = ["train", "val", "test"]
        for e in dataset:
            for current_split in random.sample(split_names, k=len(split_names)):
                # ensure we don't overfill
                if len(splits[current_split]) >= target_counts[current_split]:
                    continue

                answers = get_entity_answers(e)
                other_splits = [split for split in split_names if split != current_split]
                if all([answers_seen[split].isdisjoint(answers) for split in other_splits]):
                    splits[current_split].append(e)
                    answers_seen[current_split].update(answers)

        # Leakage checks
        assert answers_seen["train"].isdisjoint(answers_seen["val"]), "Train/Val leakage!"
        assert answers_seen["train"].isdisjoint(answers_seen["test"]), "Train/Test leakage!"
        assert answers_seen["val"].isdisjoint(answers_seen["test"]), "Val/Test leakage!"

        target_difference = calc_target_difference(splits)
        if target_difference == 0:
            return splits
        if best_try_count > target_difference:
            best_try = splits
            best_try_count = target_difference
    return best_try

dataset_split = split_no_answer_leakage(
    dataset=data,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    max_tries=300
)

100%|██████████| 300/300 [04:06<00:00,  1.22it/s]


In [21]:
dataset_dict = DatasetDict({
    "train": Dataset.from_list(dataset_split["train"]),
    "val": Dataset.from_list(dataset_split["val"]),
    "test": Dataset.from_list(dataset_split["test"]),
})

In [25]:
from collections import Counter

for split, entries in dataset_dict.items():
    counter = Counter()
    for entry in entries:
        for i in range(1, 5):
            answer = entry.get(f"granola_answer_{i}")
            if answer:
                counter[i] += 1
    print(split, counter)

train Counter({1: 6702, 2: 6582, 3: 5367, 4: 1176})
val Counter({1: 1220, 2: 1184, 3: 868, 4: 84})
test Counter({1: 1220, 2: 1170, 3: 735, 4: 60})


In [26]:
dataset_dict['train'][0]

{'id': 12642,
 'relation': 'P264',
 'question': 'What music label is On the Radio represented by?',
 'question_entity': 'On the Radio',
 'question_entity_qid': 'Q2482990',
 'question_entity_description': 'single by Natalia',
 'question_entity_popularity': 2371.0,
 'answer': 'Casablanca Records',
 'answer_entity_qid': 'Q1046707',
 'answer_entity_description': 'American record label',
 'answer_entity_popularity': 6638.0,
 'score_for_potential_error': 0.0,
 'granola_answer_1': 'Casablanca Records',
 'granola_answer_2': 'an American record label',
 'granola_answer_3': '',
 'granola_answer_4': '',
 'num_granola_answers': 2}

In [1]:
from evaluation.config import PROJECT_DIR

base_dir = f"{PROJECT_DIR}/data/granola-eq"

dataset_dict["train"].to_json(f"{base_dir}/train.jsonl")
dataset_dict["val"].to_json(f"{base_dir}/val.jsonl")
dataset_dict["test"].to_json(f"{base_dir}/test.jsonl")

NameError: name 'dataset_dict' is not defined